# 1. Regressão logística para risco de aumento
O objetivo é estimar a probabilidade de alta superior a 2% no mês-calendário seguinte; a única família de classificador treinada é regressão logística.

In [ ]:
from pathlib import Path
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown
from basket_price_prediction.config import ROOT, DATASET, SPLITS, FEATURES
from basket_price_prediction.data import load_data
from basket_price_prediction.preprocessing import prepare_monthly
from basket_price_prediction.features import build_supervised, temporal_split
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 30)

## 2. Alvo e atributos
Para cada série, o alvo é 1 somente se `(preço t+1 / preço t) - 1 > 0,02`; exatamente 2% é 0. O modelo vê preços em t, t−1 e t−2, variações, média, desvio padrão populacional, mínimo e máximo dessa janela, mês cíclico e identificadores. As três observações devem ser consecutivas. O preço futuro e a variação efetiva são apenas avaliação.

In [ ]:
raw=load_data(); monthly,cleaning=prepare_monthly(raw); supervised,report=build_supervised(monthly)
display(pd.Series(report,name="n").to_frame()); display(supervised.head())

## 3. Partição cronológica e desequilíbrio
Os limites se referem ao **mês-alvo**; as características vêm até o mês anterior. Treino termina em dezembro/2024, validação cobre janeiro–agosto/2025 e teste setembro/2025–agosto/2026.

In [ ]:
parts=temporal_split(supervised,SPLITS)
display(pd.DataFrame({k:{"n":len(v),"positivos":int(v.target.sum()),"prevalência":v.target.mean(),"primeiro alvo":v.target_month.min(),"último alvo":v.target_month.max()} for k,v in parts.items()}).T)

## 4. Pipeline e seleção
`ColumnTransformer` ajusta escala numérica e codificação one-hot somente no treino, aceitando categorias novas. São avaliados C = 0,1, 1 e 10, pesos `None` e `balanced`, e limiares de 0,10 a 0,90. O maior F1 com precisão mínima de 0,35 é escolhido na validação; se nenhum candidato atingir esse piso, usa-se o maior F1 e registra-se a exceção. Desempate por precisão e menor limiar.

## 5. Treinamento intencional e artefatos
Para gerar uma execução nova, defina `RUN_TRAINING = True` ou execute o notebook com `BASKET_TRAIN=1`. Cada execução cria um diretório único; por padrão a reabertura não treina. O modelo escolhido é reajustado em treino + validação e só então o teste é consultado. Predições de validação são do modelo anterior ao reajuste.

In [ ]:
import os
from basket_price_prediction.experiments import run_training, load_experiment
RUN_TRAINING = os.environ.get("BASKET_TRAIN", "0") == "1"
if RUN_TRAINING:
    experiment_path=run_training()
    print("Experimento concluído:",experiment_path)
else:
    print("Treinamento não solicitado. Defina RUN_TRAINING=True para criar uma execução.")

## 6. Evidências da seleção e conclusão
A tabela de candidatos é produzida apenas a partir da validação; inspecione as diferenças de precisão e recall antes de interpretar o teste no próximo notebook.

In [ ]:
if RUN_TRAINING:
    experiment=load_experiment(experiment_path)
    display(pd.Series(experiment["config"]["selection"],name="seleção").to_frame())
    display(pd.read_csv(experiment_path/"reports"/"validation_candidates.csv").sort_values("f1",ascending=False).head(10))
    display(pd.Series(experiment["metrics"]["validation_before_refit"],name="validação antes do reajuste").to_frame())

O ajuste bem-sucedido, por si só, não demonstra utilidade preditiva. Consulte o teste independente, falsos alarmes, estabilidade temporal e limitações no notebook seguinte.